In [1]:
!pip install -q langgraph langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 11.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [2]:
import os
from typing import Annotated, TypedDict
from google.colab import userdata
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key")
except Exception as e:
    raise RuntimeError(
        "Please add 'GOOGLE_API_KEY' to your Colab Secrets (Key icon on the left sidebar) "
        "and enable notebook access."
    ) from e

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

def extract_text_safely(content) -> str:
    if isinstance(content, str):
        return content
    elif isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(item["text"])
            elif isinstance(item, str):
                parts.append(item)
            else:
                parts.append(str(item))
        return "".join(parts)
    return str(content)

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str

@tool
def process_refund(user_id: str, amount: float) -> str:
    """Executes a financial refund for a specific user ID."""
    return f"SUCCESS: Refund of ${amount} has been processed for User '{user_id}'."

def supervisor_agent(state: AgentState) -> AgentState:
    system_prompt = (
        "You are a Support Router. Analyze the user prompt:\n"
        "- If it is general technical troubleshooting, respond with 'SUPPORT'.\n"
        "- If it involves financial refunds or account modifications, respond with 'ACTION'.\n"
        "Respond ONLY with 'SUPPORT' or 'ACTION'."
    )

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    raw_text = extract_text_safely(response.content)
    decision = raw_text.strip().upper()

    if "ACTION" in decision:
        next_step = "account_actions_agent"
    elif "SUPPORT" in decision:
        next_step = "tech_support_agent"
    else:
        next_step = END

    return {"next_node": next_step}

def tech_support_agent(state: AgentState) -> AgentState:
    system_prompt = SystemMessage(content="You are a helpful Technical Support Specialist. Provide concise troubleshooting guidance.")
    messages = [system_prompt] + state["messages"]
    response = llm.invoke(messages)

    text_content = extract_text_safely(response.content)
    return {"messages": [AIMessage(content=f"[Tech Support]: {text_content}")], "next_node": END}
def account_actions_agent(state: AgentState) -> AgentState:
    llm_with_tools = llm.bind_tools([process_refund])
    system_prompt = SystemMessage(content="You are an Account Manager. Use the process_refund tool to issue user refunds when requested.")

    messages = [system_prompt] + state["messages"]
    response = llm_with_tools.invoke(messages)

    if response.tool_calls:
        tool_call = response.tool_calls[0]
        tool_output = process_refund.invoke(tool_call["args"])
        final_msg = f"[Account Agent]: Executed tool. Result: {tool_output}"
    else:
        text_content = extract_text_safely(response.content)
        final_msg = f"[Account Agent]: {text_content}"

    return {"messages": [AIMessage(content=final_msg)], "next_node": END}

workflow = StateGraph(AgentState)

workflow.add_node("supervisor", supervisor_agent)
workflow.add_node("tech_support_agent", tech_support_agent)
workflow.add_node("account_actions_agent", account_actions_agent)

workflow.add_edge(START, "supervisor")

workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "tech_support_agent": "tech_support_agent",
        "account_actions_agent": "account_actions_agent",
        END: END
    }
)

workflow.add_edge("tech_support_agent", END)
workflow.add_edge("account_actions_agent", END)

app = workflow.compile()

def run_demo(user_query: str):
    print(f"\n================ USER QUERY ================\n{user_query}")
    inputs = {"messages": [HumanMessage(content=user_query)]}
    result = app.invoke(inputs)
    print("\n================ SYSTEM RESPONSE ================")
    print(result["messages"][-1].content)

run_demo("My app keeps freezing whenever I try to upload a PNG file. How can I fix this?")
run_demo("I was billed twice by mistake. Please refund $49.99 for my account 'user_9876'.")


================ USER QUERY ================
My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

================ SYSTEM RESPONSE ================
[Tech Support]: To fix the app freezing during PNG uploads, follow these troubleshooting steps:

1. **Test a Different PNG:** Try uploading a different, smaller PNG file. If it works, the original file may be corrupted or too large.
2. **Re-save or Convert the Image:** Open the PNG in an image editor (like Paint or Preview) and save it as a new PNG or JPEG. This strips out corrupted metadata that can crash the app.
3. **Check File Size and Dimensions:** Extremely high-resolution or large files can cause memory spikes. Try compressing the PNG using an online tool like TinyPNG.
4. **Clear App Cache and Restart:** 
   * **Mobile:** Force close the app, go to your phone's settings > Apps > [App Name] > Clear Cache, then reopen it.
   * **Web:** Clear your browser cache or try uploading in an Incognito/Private window.